## Inverse Kinemaics - Jacobian based methods 
- Test IK methods based on Jacobian
    - IK can be solved by multiplying Inverse Matrix of Jacobian
    - Several methods to solve inverse jacobian without singularity

Create UR environment

In [1]:
import mujoco
import mujoco_viewer # new viewer
import numpy as np
import time

import sys

In [2]:
scene_config = {

    "scene": '../assets/floor_white_gray.xml',
    "panda": "../assets/panda/robot.xml",

    "robots":{
        "robot_1":{
            "frame": {
                "pos": [1, 0, 0],
                "quat": [0.7071, 0, 0, 0.7071] # wxyz order
            },
            "body": {
                "file": "panda",
                "body_name": "base"
            }
        }
    },

}

In [ ]:
# call UR with ur scene

# model_path = "../assets/ur5e_mjcf/scene.xml"

# # declare model & data
# model = mujoco.MjModel.from_xml_path(model_path)
# data = mujoco.MjData(model)

In [ ]:
# call panda with spec generate model

sys.path.append("/home/juju/pp/pp_base_mujoco/practice_mj_spec/")
from spec_generate_scene import spec_generate_model

model = spec_generate_model(scene_config)
data = mujoco.MjData(model)

Get Mujoco Jacobian

In [5]:
def get_jac_body_name(body_name=None):

    # initialize positional & rotational jacobian
    Jacobian_p = np.zeros((3,model.nu))
    Jacobian_r = np.zeros((3,model.nu))

    # get jacobian of end-effector
    mujoco.mj_jacBody(model, data, Jacobian_p, Jacobian_r, data.body(body_name).id)
    return Jacobian_p, Jacobian_r

### Function: calculate inverse jacobian

In [6]:
def get_invjac_p_name(body_name, method='svd', sigma_threshold=0.001, damping=1.0):

    jacobian_p, jacobian_r = get_jac_body_name(body_name)

    if method=='svd':
        # get inverse jacobian with Singular Value Decomposition
        U, Sigma, V_T = np.linalg.svd(jacobian_p, compute_uv=True)

        # past implementation - not good
        # Sigma_clipped_rev = np.minimum(1 / Sigma, upper_bound)

        # suppress singularities modifying sigma
        Sigma_clipped_rev = np.zeros_like(Sigma)
        for i, value in enumerate(Sigma):
            if Sigma[i] < sigma_threshold:
                Sigma_clipped_rev[i] = 0
            else:
                Sigma_clipped_rev[i] = 1/Sigma[i]

        # inverse matrix for position jacobian
        S_rev_matrix = np.zeros((model.nu,3)) # positional dimension = 3, dof = model.nu
        for i, value in enumerate(Sigma_clipped_rev):
            S_rev_matrix[i,i] = value
        J_p_inverse = V_T.T @ S_rev_matrix @ U.T

    if method=='DLS':
        # apply damped least squares
        pass
    
    return J_p_inverse

In [7]:
"""
Parameter tuning
- sigma threshold should not be big
    - if too big, robot will not move if it should move large to reach
    - small sigma = small gain = move large to get to goal
"""

'\nParameter tuning\n- sigma threshold should not be big\n    - if too big, robot will not move if it should move large to reach\n    - small sigma = small gain = move large to get to goal\n'

### MAIN loop: calculate error & update with forward

In [ ]:
""" MAIN LOOP """

# create python viewer object
viewer = mujoco_viewer.MujocoViewer(model, data)

# initialize robot
init_qpos = [3.14, -0.8, -2.5, -2.2, 0.0, 0.0, 0.0]

mujoco.mj_resetData(model, data)
data.qpos = init_qpos
mujoco.mj_forward(model, data) # first forward to get jacobian with no error

# goal position
goal_pos = [0.3, 0.3, 0.2]
# body_name = "wrist_3_link" # for ur
body_name = 'robot_1-right_hand' # for panda

# scale down error
alpha = 0.02


# # empty viewer
# for i in range(700):
#     if viewer.is_alive:

#         mujoco.mj_forward(model, data)
#         viewer.render()


while True:
    if viewer.is_alive:

        # add sphere to goal pose

        # get inverse jacobian & unit error vector
        J_p_inverse = get_invjac_p_name(body_name=body_name, method='svd')
        error = goal_pos - data.body(body_name).xpos.copy()
        print(f"current error: {error}")

        dq = alpha * (J_p_inverse @ error)
        print(f"dq: {dq}")
        data.qpos += dq
        # print(f"qpos before update: {qpos_before} \n qpos after update: {data.qpos}")

        mujoco.mj_forward(model, data)

        print(f"current body pos: {data.body(body_name).xpos}")

        # terminalize
        if np.linalg.norm(goal_pos - data.body(body_name).xpos) < 0.02:
            print("IK done.")
            break

        viewer.render()

    else:
        break



# close
viewer.close()

current error: [-0.5187308   0.02389732 -0.06179015]
dq: [ 1.89399487e-02 -1.44449551e-02  9.87671134e-03  2.53178321e-02
  2.86261691e-03  1.05751417e-03 -1.72475742e-18]
current body pos: [0.80833353 0.27641925 0.26059485]
current error: [-0.50833353  0.02358075 -0.06059485]
dq: [ 1.84957244e-02 -1.43878400e-02  9.18492737e-03  2.58367535e-02
  2.66303497e-03  1.08436235e-03  4.27210880e-18]
current body pos: [0.7981492  0.27673903 0.25942088]
current error: [-0.4981492   0.02326097 -0.05942088]
dq: [ 1.80741891e-02 -1.43106953e-02  8.53808121e-03  2.62931984e-02
  2.48094849e-03  1.12918009e-03  3.49250331e-18]
current body pos: [0.78817338 0.27706146 0.25826786]
current error: [-0.48817338  0.02293854 -0.05826786]
dq: [ 1.76740225e-02 -1.42157386e-02  7.93493009e-03  2.66940523e-02
  2.31531868e-03  1.19180894e-03  1.43233161e-18]
current body pos: [0.77840164 0.27738599 0.25713539]
current error: [-0.47840164  0.02261401 -0.05713539]
dq: [ 1.72938059e-02 -1.41050459e-02  7.3738470